# Llama 4 Maverick (via Parley) vs. the Paper's Benchmark

The paper (`notebooks/plots.ipynb`) reports an **accuracy** metric for 10 models on the real
`matharena_proofs` benchmark: the fraction of a model's proofs judged **both correct** (right
final answer) **and complete** (no missing/skipped steps) by an LLM judge, using the exact
`answer_checker` / `completeness_checker` prompts in `configs/processors/`.

This notebook computes the same metric for **Llama 4 Maverick via Parley** on a small *real*
sample of the same benchmark (pulled live from the public `Anon987281293/ProofRank` HF dataset),
using the paper's own grading prompts, then compares it to the paper's table.

**Caveats — this is an indicative comparison, not a replication:**
- Sample size here is ~12 problems, vs. the paper's full benchmark set.
- Single attempt per problem (the paper's solver config allows up to 10 retries on rejected
  generations; we don't reject/retry here).
- The judge model here is `bedrock/claude-haiku-4-5` (cheap, via Parley) rather than the paper's
  own judge — so treat this as a reasonable proxy, not an identical grading condition.

## 1. Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.expanduser("~/Desktop/code_mit_parley_api/.env"))
assert "PARLEY_API_KEY" in os.environ, "PARLEY_API_KEY not found -- check the .env path above"
print("PARLEY_API_KEY loaded:", os.environ["PARLEY_API_KEY"][:14] + "...")

PARLEY_API_KEY loaded: sk-parley-v1-y...


## 2. Pull a real sample from the benchmark dataset

`matharena_proofs`'s raw problems (with gold answers) were pushed to the public HF dataset
`Anon987281293/ProofRank` (see `scripts/data/submit_hf_dataset.py`) as the `main` split. We sample
12 of them with a fixed seed for reproducibility.

In [2]:
from datasets import load_dataset

ds = load_dataset("Anon987281293/ProofRank")["main"]
sample = ds.shuffle(seed=42).select(range(12))

problems = [
    {"problem_id": row["problem_id"], "problem": row["problem"], "gold_answer": row["gold_answer"]}
    for row in sample
]
for p in problems:
    print(p["problem_id"])

imo-bench_number_theory-094
imo-bench_geometry-099
imo-bench_combinatorics-037
hmmt_feb_2025_9
imo-bench_geometry-094
apex_2025_2
imo-bench_combinatorics-081
imo-bench_combinatorics-077
imo-bench_geometry-054
apex_2025_9
shortlist_2025_9
imo-bench_number_theory-002


In [3]:
import json, os

os.makedirs("../data/raw/parley_comparison", exist_ok=True)
with open("../data/raw/parley_comparison/sample.json", "w") as f:
    json.dump(problems, f, indent=4)
print("Wrote", len(problems), "problems to data/raw/parley_comparison/sample.json")

Wrote 12 problems to data/raw/parley_comparison/sample.json


## 3. Config for this run

`configs/projects/parley_comparison.yaml` and `configs/solvers/parley_comparison.yaml` mirror the
real `matharena_proofs` project/solver configs (same prompt, same use of `gold_answer`), pointed
at the `parley/llama-4-maverick` model config from the smoke-test notebook. One attempt, one
solution per problem -- this is a comparison spot-check, not a full benchmark run.

In [4]:
!rm -rf ../data/unsolved/parley_comparison ../data/solved/parley_comparison ../data/postprocess/parley_comparison
!cd .. && python scripts/process.py --project parley_comparison
!find ../data/unsolved/parley_comparison -type f | sort

../data/unsolved/parley_comparison/apex/2025_2.json
../data/unsolved/parley_comparison/apex/2025_9.json
../data/unsolved/parley_comparison/hmmt/feb_2025_9.json
../data/unsolved/parley_comparison/imo-bench/combinatorics-037.json
../data/unsolved/parley_comparison/imo-bench/combinatorics-077.json
../data/unsolved/parley_comparison/imo-bench/combinatorics-081.json
../data/unsolved/parley_comparison/imo-bench/geometry-054.json
../data/unsolved/parley_comparison/imo-bench/geometry-094.json
../data/unsolved/parley_comparison/imo-bench/geometry-099.json
../data/unsolved/parley_comparison/imo-bench/number_theory-002.json
../data/unsolved/parley_comparison/imo-bench/number_theory-094.json
../data/unsolved/parley_comparison/shortlist/2025_9.json


## 4. Solve all 12 problems via Parley / Llama 4 Maverick

In [5]:
!cd .. && python scripts/solve.py --project parley_comparison --synchronous

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Found 12 files to run.
Running 12 problems for project parley_comparison
2026-09-17 15:28:32.595 | INFO     | proofrank.solve:solve:32 - Initializing APIQuery for parley/llama-4-maverick...
2026-09-17 15:28:32.597 | INFO     | proofrank.api:run_queries:430 - Running 12 queries.
  0%|                                                   | 0/12 [00:00<?, ?it/s]

  8%|███▌                                       | 1/12 [00:06<01:11,  6.51s/it]

2026-09-17 15:28:39.982 | INFO     | proofrank.api:openai_query_no_response:1296 - Got OpenAI error: Error code: 502 - {'error': {'message': 'Bedrock API error: Model has timed out in processing the request. Try your request again.', 'type': 'server_error', 'param': None, 'code': 'provider_error'}}


 17%|███████▏                                   | 2/12 [00:08<00:36,  3.67s/it]

...

 58%|█████████████████████████                  | 7/12 [00:17<00:08,  1.80s/it]

 67%|████████████████████████████▋              | 8/12 [00:24<00:13,  3.42s/it]

2026-09-17 15:28:57.667 | INFO     | proofrank.solve:run_problem:136 - Processed 10 problems with parley/llama-4-maverick, total cost: 0.0003382
 83%|███████████████████████████████████       | 10/12 [00:25<00:04,  2.04s/it]

 92%|██████████████████████████████████████▌   | 11/12 [00:30<00:03,  3.00s/it]

100%|██████████████████████████████████████████| 12/12 [01:15<00:00,  6.30s/it]
2026-09-17 15:29:48.269 | INFO     | proofrank.solve:single_loop:111 - Total cost for parley/llama-4-maverick: 0.00040708
Total cost: 0.00 USD


In [6]:
!cd .. && python scripts/postprocess.py --project parley_comparison

100%|████████████████████████████████████████| 12/12 [00:00<00:00, 3904.10it/s]
Total samples: 12
Test samples: 12
Model counts:
parley/llama-4-maverick: 12


## 5. Grade each solution with the paper's exact judge prompts

Loading the real prompt templates from `configs/processors/answer_checker.yaml` and
`completeness_checker.yaml` (not re-typing them), formatting with each problem's `problem`,
`solution`, and `gold_answer`, and parsing the `\boxed{{...}}` verdict exactly the way
`scripts/results/correctness_eval.py` does (`"incorrect" not in output.lower()` /
`"incomplete" not in output.lower()`).

In [7]:
import yaml

with open("../configs/processors/answer_checker.yaml") as f:
    answer_checker_prompt = yaml.safe_load(f)["prompt"]
with open("../configs/processors/completeness_checker.yaml") as f:
    completeness_checker_prompt = yaml.safe_load(f)["prompt"]

samples = json.load(open("../data/postprocess/parley_comparison/test_samples.json"))
print(f"Loaded {len(samples)} solved samples to grade.")

Loaded 12 solved samples to grade.


In [8]:
from proofrank.api import APIQuery

JUDGE_MODEL = "bedrock/claude-haiku-4-5"

answer_queries = [
    answer_checker_prompt.format(
        problem=s["problem"], solution=s["solution"], gold_answer=s["gold_answer"]
    )
    for s in samples
]
completeness_queries = [
    completeness_checker_prompt.format(problem=s["problem"], solution=s["solution"])
    for s in samples
]

judge = APIQuery(model=JUDGE_MODEL, api="parley", max_tokens=2000)

def extract_text(output):
    if isinstance(output, list):
        return output[-1]["content"]
    return output

answer_outputs = {}
judge_cost = 0.0
for idx, output, cost in judge.run_queries(answer_queries):
    answer_outputs[idx] = extract_text(output)
    judge_cost += cost["cost"]

completeness_outputs = {}
for idx, output, cost in judge.run_queries(completeness_queries):
    completeness_outputs[idx] = extract_text(output)
    judge_cost += cost["cost"]

print(f"Judging cost: ${judge_cost:.4f}")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


2026-09-17 15:29:52.564 | INFO     | proofrank.api:run_queries:430 - Running 12 queries.


  0%|          | 0/12 [00:00<?, ?it/s]

  8%|▊         | 1/12 [00:03<00:39,  3.63s/it]

 17%|█▋        | 2/12 [00:03<00:16,  1.69s/it]

...

 75%|███████▌  | 9/12 [00:09<00:01,  2.15it/s]

 83%|████████▎ | 10/12 [00:09<00:00,  2.61it/s]

100%|██████████| 12/12 [00:09<00:00,  4.15it/s]

100%|██████████| 12/12 [00:09<00:00,  1.21it/s]

Judging cost: $0.0768


In [9]:
results = []
for i, s in enumerate(samples):
    is_correct = "incorrect" not in answer_outputs[i].lower()
    is_complete = "incomplete" not in completeness_outputs[i].lower()
    results.append({
        "problem_id": s["problem_id"],
        "correct": is_correct,
        "complete": is_complete,
        "accurate": is_correct and is_complete,
    })

for r in results:
    print(f"{r['problem_id']:20s} correct={r['correct']!s:5s} complete={r['complete']!s:5s} "
          f"-> accurate={r['accurate']}")

accuracy = 100 * sum(r["accurate"] for r in results) / len(results)
print(f"\nLlama 4 Maverick accuracy on this {len(results)}-problem sample: {accuracy:.1f}%")

hmmt_feb_2025_9      correct=False complete=False -> accurate=False
apex_2025_9          correct=False complete=True  -> accurate=False
apex_2025_2          correct=False complete=False -> accurate=False
imo-bench_geometry-099 correct=True  complete=False -> accurate=False
imo-bench_combinatorics-037 correct=False complete=False -> accurate=False
imo-bench_geometry-094 correct=False complete=False -> accurate=False
imo-bench_combinatorics-077 correct=False complete=False -> accurate=False
imo-bench_number_theory-094 correct=False complete=False -> accurate=False
imo-bench_number_theory-002 correct=False complete=False -> accurate=False
imo-bench_combinatorics-081 correct=False complete=False -> accurate=False
imo-bench_geometry-054 correct=False complete=False -> accurate=False
shortlist_2025_9     correct=True  complete=False -> accurate=False

Llama 4 Maverick accuracy on this 12-problem sample: 0.0%


## 6. Compare against the paper's benchmark table

The accuracy values below for the other 10 models are copied verbatim from the `elos` dict in
`notebooks/plots.ipynb` (column index 27, `"accuracy"`) -- the real, full-benchmark numbers from
the paper.

In [10]:
paper_accuracy = {
    "GPT-5.4": 86.4,
    "Gemini-3.1-Pro": 60.7,
    "Kimi-2.5-Think": 53.6,
    "GLM-5": 49.1,
    "DeepSeek-v3.2": 47.9,
    "StepFun-3.5": 41.1,
    "OSS-120B": 34.3,
    "Grok-4.1-Fast": 31.4,
    "Gemini-3-Flash": 29.8,
    "Qwen3.5-397B": 29.3,
}

all_scores = dict(paper_accuracy)
all_scores[f"Llama-4-Maverick (Parley, n={len(results)})"] = accuracy

for model, score in sorted(all_scores.items(), key=lambda kv: -kv[1]):
    marker = "  <-- this run" if "Maverick" in model else ""
    print(f"{model:32s} {score:5.1f}%{marker}")

GPT-5.4                           86.4%
Gemini-3.1-Pro                    60.7%
Kimi-2.5-Think                    53.6%
GLM-5                             49.1%
DeepSeek-v3.2                     47.9%
StepFun-3.5                       41.1%
OSS-120B                          34.3%
Grok-4.1-Fast                     31.4%
Gemini-3-Flash                    29.8%
Qwen3.5-397B                      29.3%
Llama-4-Maverick (Parley, n=12)    0.0%  <-- this run


## Summary

- Ran the real `matharena_proofs` accuracy metric -- the paper's own `answer_checker` +
  `completeness_checker` judge prompts -- on 12 real problems sampled from the public
  `Anon987281293/ProofRank` dataset, solved via Parley's free `bedrock/llama-4-maverick-17b`.
- Result printed above alongside the paper's 10-model table for direct comparison.
- Given the caveats in section 1 (small sample, single attempt, different/cheaper judge model),
  treat this as an indicative data point for where Llama 4 Maverick lands relative to the paper's
  benchmark -- not a rigorous, apples-to-apples replication of the paper's full evaluation.